# WikiArt Test 13 - Style + Artist (Warm-start from Test 12)

This notebook trains a single model that predicts both style and artist.

## What This Notebook Does
1. Loads the Test 12 style checkpoint as warm-start.
2. Adds a new artist head.
3. Trains artist head first while style backbone is frozen.
4. Optionally fine-tunes the last style blocks using style-distillation so style quality is preserved.
5. Reports style and artist metrics and updates summary CSV.

## Why This Matches Your Goal
- You do not retrain style from scratch.
- You still end with one shared model that can predict both style and artist.
- Training flow is split into clear stages so it is easy to understand and edit.

In [1]:
# --- Imports + configuration -------------------------------------------------
from __future__ import annotations

import datetime
import math
import os
import random
from copy import deepcopy
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

import timm
from timm.utils import ModelEmaV2


@dataclass
class TrainConfig:
    model_name: str = "vit_base_patch14_reg4_dinov2.lvd142m"
    image_size: int = 448

    batch_size: int = 8
    effective_batch_size: int = 64
    num_workers: int = 0

    # Stage 1: artist head only
    artist_head_only_epochs: int = 10
    artist_head_lr: float = 6e-4

    # Stage 2: optional limited fine-tune
    run_stage2_finetune: bool = True
    finetune_epochs: int = 20
    unfreeze_last_blocks: int = 2
    finetune_backbone_lr: float = 5e-6
    finetune_artist_head_lr: float = 2e-5

    # Keep style behavior during stage 2
    style_distill_weight: float = 1.0
    style_distill_temperature: float = 2.0

    warmup_epochs: int = 3
    weight_decay: float = 0.03
    label_smoothing: float = 0.04
    grad_clip: float = 1.0
    ema_decay: float = 0.9999
    seed: int = 42

    # Warm-start from test12 style checkpoint
    warmstart_style_checkpoint: str = "models/wikiart_test12_vitlarge_448_best.pt"

    output_dir: str = "models"
    model_save_name: str = "wikiart_test13_style_artist_warmstart_best.pt"
    history_csv: str = "models/results/wikiart_test13_style_artist_history.csv"
    summary_csv: str = "models/results/wikiart_tests_1_to_13_summary.csv"
    previous_summary_csv: str = "models/results/wikiart_tests_1_to_12_summary.csv"

    @property
    def accumulate_steps(self) -> int:
        return max(1, self.effective_batch_size // self.batch_size)

    @property
    def total_epochs(self) -> int:
        stage2 = self.finetune_epochs if self.run_stage2_finetune else 0
        return self.artist_head_only_epochs + stage2


cfg = TrainConfig()
print(cfg)
print(f"Gradient accumulation steps: {cfg.accumulate_steps}")
print(f"Total epochs: {cfg.total_epochs}")

TrainConfig(model_name='vit_base_patch14_reg4_dinov2.lvd142m', image_size=448, batch_size=8, effective_batch_size=64, num_workers=0, artist_head_only_epochs=10, artist_head_lr=0.0006, run_stage2_finetune=True, finetune_epochs=20, unfreeze_last_blocks=2, finetune_backbone_lr=5e-06, finetune_artist_head_lr=2e-05, style_distill_weight=1.0, style_distill_temperature=2.0, warmup_epochs=3, weight_decay=0.03, label_smoothing=0.04, grad_clip=1.0, ema_decay=0.9999, seed=42, warmstart_style_checkpoint='models/wikiart_test12_vitlarge_448_best.pt', output_dir='models', model_save_name='wikiart_test13_style_artist_warmstart_best.pt', history_csv='models/results/wikiart_test13_style_artist_history.csv', summary_csv='models/results/wikiart_tests_1_to_13_summary.csv', previous_summary_csv='models/results/wikiart_tests_1_to_12_summary.csv')
Gradient accumulation steps: 8
Total epochs: 30


## Data Utilities
These helpers find project paths, load label CSV files, and build the validation/test splits used for evaluation.

In [2]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def discover_project_root() -> Path:
    root = Path(".").resolve()
    for _ in range(6):
        if (root / "datasets").exists():
            return root
        root = root.parent
    raise FileNotFoundError("Could not find project root (no datasets directory found)")


def discover_paths(project_root: Path) -> Dict[str, Path]:
    wikiart_dir = project_root / "datasets" / "Wikiart"
    paths = {
        "wikiart_dir": wikiart_dir,
        "style_train": wikiart_dir / "style_train.csv",
        "style_val": wikiart_dir / "style_val.csv",
        "artist_train": wikiart_dir / "artist_train.csv",
        "artist_val": wikiart_dir / "artist_val.csv",
    }
    missing = [str(p) for p in paths.values() if isinstance(p, Path) and not p.exists()]
    if missing:
        raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))
    return paths


def load_label_csv(path: Path, label_name: str) -> pd.DataFrame:
    df = pd.read_csv(path, header=None, names=["relative_path", label_name])
    df[label_name] = df[label_name].astype(int)
    return df


def filter_existing_rows(df: pd.DataFrame, image_root: Path, split_name: str) -> pd.DataFrame:
    keep = df["relative_path"].apply(lambda p: (image_root / p).exists())
    n_missing = int((~keep).sum())
    if n_missing > 0:
        print(f"[{split_name}] Skipping {n_missing} missing files")
    return df.loc[keep].reset_index(drop=True)


def stratified_half_split(df: pd.DataFrame, label_col: str, seed: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    parts: List[pd.DataFrame] = []
    for _, grp in df.groupby(label_col, sort=False):
        if len(grp) > 1:
            parts.append(grp.sample(frac=0.5, random_state=seed))
        else:
            parts.append(grp)

    eval_test = pd.concat(parts, axis=0)
    eval_val = df.drop(index=eval_test.index)
    if len(eval_val) == 0:
        eval_val = df.sample(frac=0.5, random_state=seed)
        eval_test = df.drop(index=eval_val.index)
    return eval_val.reset_index(drop=True), eval_test.reset_index(drop=True)


set_seed(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## Dataset, Model, and DataLoaders
This section defines the shared style backbone plus artist head, and builds all data loaders.

In [3]:
class WikiArtLabelDataset(Dataset):
    def __init__(self, rows: pd.DataFrame, image_root: Path, label_col: str, transform=None):
        self.rows = rows.reset_index(drop=True).copy()
        self.image_root = Path(image_root)
        self.label_col = label_col
        self.transform = transform

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int):
        row = self.rows.iloc[idx]
        image = Image.open(self.image_root / row["relative_path"]).convert("RGB")
        label = int(row[self.label_col])
        if self.transform is not None:
            image = self.transform(image)
        return image, label


class StyleArtistModel(nn.Module):
    def __init__(self, model_name: str, image_size: int, n_style: int, n_artist: int):
        super().__init__()
        self.style_model = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=n_style,
            img_size=image_size,
        )
        feat_dim = self.style_model.num_features
        self.artist_head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(p=0.2),
            nn.Linear(feat_dim, n_artist),
        )

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.style_model.forward_features(x)
        feats = self.style_model.forward_head(feats, pre_logits=True)
        return feats

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        feats = self.extract_features(x)
        style_input = self.style_model.head_drop(feats) if hasattr(self.style_model, "head_drop") else feats
        style_logits = self.style_model.head(style_input)
        artist_logits = self.artist_head(feats)
        return style_logits, artist_logits


def build_transforms(image_size: int):
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]
    train_tfms = transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.55, 1.0), ratio=(0.75, 1.33)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandAugment(num_ops=2, magnitude=11),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.04),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
        transforms.RandomErasing(p=0.20, scale=(0.02, 0.18), ratio=(0.3, 3.3), value="random"),
    ])
    eval_tfms = transforms.Compose([
        transforms.Resize(int(image_size * 1.14)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])
    return train_tfms, eval_tfms


def pick_num_workers(requested: int) -> int:
    if requested < 0:
        return max(0, (os.cpu_count() or 1) - 1)
    return requested


def make_loaders(
    cfg: TrainConfig,
    device: torch.device,
    wikiart_dir: Path,
    style_val_df: pd.DataFrame,
    artist_train_df: pd.DataFrame,
    artist_val_df: pd.DataFrame,
) -> Dict[str, DataLoader]:
    train_tfms, eval_tfms = build_transforms(cfg.image_size)
    num_workers = pick_num_workers(cfg.num_workers)
    pin_mem = device.type == "cuda"
    loader_kw = dict(
        num_workers=num_workers,
        pin_memory=pin_mem,
        persistent_workers=(num_workers > 0),
    )

    style_eval_val_df, style_eval_test_df = stratified_half_split(style_val_df, "style_label", cfg.seed)
    artist_eval_val_df, artist_eval_test_df = stratified_half_split(artist_val_df, "artist_label", cfg.seed)

    artist_train_ds = WikiArtLabelDataset(artist_train_df, wikiart_dir, "artist_label", transform=train_tfms)
    style_val_ds = WikiArtLabelDataset(style_eval_val_df, wikiart_dir, "style_label", transform=eval_tfms)
    style_test_ds = WikiArtLabelDataset(style_eval_test_df, wikiart_dir, "style_label", transform=eval_tfms)
    artist_val_ds = WikiArtLabelDataset(artist_eval_val_df, wikiart_dir, "artist_label", transform=eval_tfms)
    artist_test_ds = WikiArtLabelDataset(artist_eval_test_df, wikiart_dir, "artist_label", transform=eval_tfms)

    loaders = {
        "artist_train": DataLoader(
            artist_train_ds,
            batch_size=cfg.batch_size,
            shuffle=True,
            drop_last=True,
            **loader_kw,
        ),
        "style_val": DataLoader(style_val_ds, batch_size=cfg.batch_size, shuffle=False, **loader_kw),
        "style_test": DataLoader(style_test_ds, batch_size=cfg.batch_size, shuffle=False, **loader_kw),
        "artist_val": DataLoader(artist_val_ds, batch_size=cfg.batch_size, shuffle=False, **loader_kw),
        "artist_test": DataLoader(artist_test_ds, batch_size=cfg.batch_size, shuffle=False, **loader_kw),
    }

    print(f"Artist train samples: {len(artist_train_ds):,} (batches={len(loaders['artist_train'])})")
    print(f"Style val/test: {len(style_val_ds):,} / {len(style_test_ds):,}")
    print(f"Artist val/test: {len(artist_val_ds):,} / {len(artist_test_ds):,}")
    return loaders


def load_style_warmstart(model: StyleArtistModel, ckpt_path: Path) -> None:
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Warm-start checkpoint not found: {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location="cpu")
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        ckpt = ckpt["state_dict"]
    if not isinstance(ckpt, dict):
        raise RuntimeError("Warm-start checkpoint format not recognized")

    ckpt = {k.replace("module.", ""): v for k, v in ckpt.items()}
    model_state = model.style_model.state_dict()
    compatible = {
        k: v
        for k, v in ckpt.items()
        if k in model_state and model_state[k].shape == v.shape
    }
    loaded_params = sum(v.numel() for v in compatible.values())
    total_params = sum(v.numel() for v in model_state.values())
    coverage = loaded_params / max(1, total_params)

    missing, unexpected = model.style_model.load_state_dict(compatible, strict=False)
    print(f"Warm-start loaded: missing={len(missing)} unexpected={len(unexpected)} coverage={coverage*100:.2f}%")
    if coverage < 0.90:
        raise RuntimeError(
            "Warm-start checkpoint coverage too low. Check model architecture mismatch with Test 12."
        )


def freeze_all_style_params(model: StyleArtistModel) -> None:
    for p in model.style_model.parameters():
        p.requires_grad = False
    for p in model.artist_head.parameters():
        p.requires_grad = True


def unfreeze_last_style_blocks(model: StyleArtistModel, n_blocks: int) -> None:
    freeze_all_style_params(model)
    if n_blocks <= 0 or not hasattr(model.style_model, "blocks"):
        return
    blocks = list(model.style_model.blocks)
    n = min(n_blocks, len(blocks))
    for block in blocks[-n:]:
        for p in block.parameters():
            p.requires_grad = True
    for name, module in model.style_model.named_modules():
        if name.startswith("norm") or name.startswith("fc_norm"):
            for p in module.parameters(recurse=False):
                p.requires_grad = True

## Training and Evaluation Helpers
These helpers train artist with two stages and evaluate both tasks with TTA where needed.

In [4]:
def build_stage2_optimizer(model: StyleArtistModel, cfg: TrainConfig) -> torch.optim.Optimizer:
    backbone_params = [p for p in model.style_model.parameters() if p.requires_grad]
    artist_params = [p for p in model.artist_head.parameters() if p.requires_grad]
    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": cfg.finetune_backbone_lr, "weight_decay": cfg.weight_decay})
    if artist_params:
        groups.append({"params": artist_params, "lr": cfg.finetune_artist_head_lr, "weight_decay": cfg.weight_decay})
    if not groups:
        raise RuntimeError("No trainable parameters found for stage 2")
    return torch.optim.AdamW(groups, betas=(0.9, 0.999), eps=1e-8)


def build_scheduler(
    optimizer: torch.optim.Optimizer,
    warmup_steps: int,
    total_steps: int,
) -> torch.optim.lr_scheduler._LRScheduler:
    if total_steps <= 0:
        raise ValueError("total_steps must be > 0")
    warmup_steps = max(0, min(warmup_steps, max(0, total_steps - 1)))
    if warmup_steps == 0:
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=0.0)
    warmup_sched = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=0.05,
        end_factor=1.0,
        total_iters=warmup_steps,
    )
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=total_steps - warmup_steps,
        eta_min=0.0,
    )
    return torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_sched, cosine_sched],
        milestones=[warmup_steps],
    )


@torch.no_grad()
def topk_accuracy(logits: torch.Tensor, targets: torch.Tensor, k: int = 1) -> float:
    _, pred = logits.topk(k, dim=1, largest=True, sorted=True)
    correct = pred.eq(targets.view(-1, 1).expand_as(pred))
    return correct[:, :k].reshape(-1).float().sum().item() / max(1, targets.size(0))


@torch.no_grad()
def evaluate_task(
    model: StyleArtistModel,
    loader: DataLoader,
    task: str,
    criterion: nn.Module,
    device: torch.device,
    num_classes: int,
    tta_flip: bool = False,
    tta_scales: Optional[List[float]] = None,
) -> Dict[str, float]:
    if task not in {"style", "artist"}:
        raise ValueError("task must be style or artist")
    model.eval()
    total_loss, total_top1, total_top5, n = 0.0, 0.0, 0.0, 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        bs = labels.size(0)
        h, w = images.shape[-2:]

        views = [images]
        if tta_flip:
            views.append(images.flip(-1))
        if tta_scales:
            for s in tta_scales:
                new_h, new_w = int(h * s), int(w * s)
                scaled = F.interpolate(images, size=(new_h, new_w), mode="bicubic", align_corners=False)
                y0 = max(0, (new_h - h) // 2)
                x0 = max(0, (new_w - w) // 2)
                scaled = scaled[:, :, y0 : y0 + min(new_h, h), x0 : x0 + min(new_w, w)]
                if scaled.shape[-2:] != (h, w):
                    scaled = F.interpolate(scaled, size=(h, w), mode="bicubic", align_corners=False)
                views.append(scaled)
                if tta_flip:
                    views.append(scaled.flip(-1))

        logits_sum = None
        for view in views:
            style_logits, artist_logits = model(view)
            logits = style_logits if task == "style" else artist_logits
            logits_sum = logits if logits_sum is None else logits_sum + logits
        logits_avg = logits_sum / len(views)

        loss = criterion(logits_avg, labels)
        total_loss += float(loss.item()) * bs
        total_top1 += topk_accuracy(logits_avg, labels, k=1) * bs
        total_top5 += topk_accuracy(logits_avg, labels, k=min(5, num_classes)) * bs
        n += bs

    return {
        "loss": total_loss / max(1, n),
        "top1": total_top1 / max(1, n),
        "top5": total_top5 / max(1, n),
        "n": float(n),
    }


def kl_distill_loss(student_logits: torch.Tensor, teacher_logits: torch.Tensor, temperature: float) -> torch.Tensor:
    t = max(1e-6, float(temperature))
    s = F.log_softmax(student_logits / t, dim=1)
    q = F.softmax(teacher_logits / t, dim=1)
    return F.kl_div(s, q, reduction="batchmean") * (t * t)


def train_artist_epoch(
    model: StyleArtistModel,
    teacher_model: Optional[StyleArtistModel],
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler: Optional[torch.optim.lr_scheduler._LRScheduler],
    scaler: torch.amp.GradScaler,
    ema_model: Optional[ModelEmaV2],
    cfg: TrainConfig,
    device: torch.device,
    stage_name: str,
) -> Dict[str, float]:
    model.train()
    optimizer.zero_grad()

    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    total_loss = 0.0
    total_artist_loss = 0.0
    total_distill_loss = 0.0
    total_acc = 0.0
    n_seen = 0

    pbar = tqdm(loader, desc=stage_name, leave=False)
    for step, (images, artist_labels) in enumerate(pbar, start=1):
        images = images.to(device, non_blocking=True)
        artist_labels = artist_labels.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
            style_logits, artist_logits = model(images)
            artist_loss = criterion(artist_logits, artist_labels)

            distill = torch.zeros((), device=device)
            if teacher_model is not None and cfg.style_distill_weight > 0:
                with torch.no_grad():
                    teacher_style_logits, _ = teacher_model(images)
                distill = kl_distill_loss(style_logits, teacher_style_logits, cfg.style_distill_temperature)

            loss_full = artist_loss + cfg.style_distill_weight * distill
            loss = loss_full / cfg.accumulate_steps

        scaler.scale(loss).backward()

        if step % cfg.accumulate_steps == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if scheduler is not None:
                scheduler.step()
            if ema_model is not None:
                ema_model.update(model)

        bs = artist_labels.size(0)
        n_seen += bs
        total_loss += float(loss_full.item()) * bs
        total_artist_loss += float(artist_loss.item()) * bs
        total_distill_loss += float(distill.item()) * bs
        total_acc += topk_accuracy(artist_logits.detach(), artist_labels, k=1) * bs

        pbar.set_postfix(loss=f"{total_loss/max(1,n_seen):.4f}", artist=f"{total_acc/max(1,n_seen):.4f}")

    return {
        "loss": total_loss / max(1, n_seen),
        "artist_loss": total_artist_loss / max(1, n_seen),
        "distill_loss": total_distill_loss / max(1, n_seen),
        "artist_top1": total_acc / max(1, n_seen),
    }


def append_summary_row(summary_path: Path, previous_summary_path: Path, row: Dict) -> None:
    if summary_path.exists():
        df = pd.read_csv(summary_path)
    elif previous_summary_path.exists():
        df = pd.read_csv(previous_summary_path)
    else:
        df = pd.DataFrame()

    experiment_name = str(row.get("experiment", "")).lower()
    if not df.empty and "experiment" in df.columns:
        df = df[df["experiment"].astype(str).str.lower() != experiment_name]

    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(summary_path, index=False)

## Run Test 13
This final cell runs the full Test 13 pipeline and writes checkpoint/history/summary outputs.

In [5]:
def run_test13(cfg: TrainConfig) -> None:
    project_root = discover_project_root()
    paths = discover_paths(project_root)
    wikiart_dir = paths["wikiart_dir"]

    style_train_df = filter_existing_rows(load_label_csv(paths["style_train"], "style_label"), wikiart_dir, "style_train")
    style_val_df = filter_existing_rows(load_label_csv(paths["style_val"], "style_label"), wikiart_dir, "style_val")
    artist_train_df = filter_existing_rows(load_label_csv(paths["artist_train"], "artist_label"), wikiart_dir, "artist_train")
    artist_val_df = filter_existing_rows(load_label_csv(paths["artist_val"], "artist_label"), wikiart_dir, "artist_val")

    n_style = int(max(style_train_df["style_label"].max(), style_val_df["style_label"].max()) + 1)
    n_artist = int(max(artist_train_df["artist_label"].max(), artist_val_df["artist_label"].max()) + 1)
    print(f"Classes -> style: {n_style}, artist: {n_artist}")

    loaders = make_loaders(
        cfg=cfg,
        device=device,
        wikiart_dir=wikiart_dir,
        style_val_df=style_val_df,
        artist_train_df=artist_train_df,
        artist_val_df=artist_val_df,
    )

    model = StyleArtistModel(
        model_name=cfg.model_name,
        image_size=cfg.image_size,
        n_style=n_style,
        n_artist=n_artist,
    ).to(device)

    warmstart_path = project_root / cfg.warmstart_style_checkpoint
    print(f"Loading warm-start: {warmstart_path}")
    load_style_warmstart(model, warmstart_path)

    teacher_model = deepcopy(model).to(device).eval()
    for p in teacher_model.parameters():
        p.requires_grad = False

    ema_model = ModelEmaV2(model, decay=cfg.ema_decay, device=device)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    artist_eval_criterion = nn.CrossEntropyLoss()
    style_eval_criterion = nn.CrossEntropyLoss()

    output_ckpt_path = project_root / cfg.output_dir / cfg.model_save_name
    output_ckpt_path.parent.mkdir(parents=True, exist_ok=True)

    history: List[Dict] = []
    best_artist_val = -1.0
    best_style_val = -1.0
    best_epoch = 0

    # Stage 1
    print("\n" + "=" * 72)
    print("Stage 1 - Train artist head only")
    print("=" * 72)
    freeze_all_style_params(model)
    optimizer_s1 = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg.artist_head_lr,
        weight_decay=cfg.weight_decay,
    )
    total_steps_s1 = max(1, cfg.artist_head_only_epochs * len(loaders["artist_train"]) // cfg.accumulate_steps)
    warmup_s1 = cfg.warmup_epochs * len(loaders["artist_train"]) // cfg.accumulate_steps
    scheduler_s1 = build_scheduler(optimizer_s1, warmup_steps=warmup_s1, total_steps=total_steps_s1)

    global_epoch = 0
    for _ in range(cfg.artist_head_only_epochs):
        global_epoch += 1
        train_metrics = train_artist_epoch(
            model=model,
            teacher_model=None,
            loader=loaders["artist_train"],
            optimizer=optimizer_s1,
            scheduler=scheduler_s1,
            scaler=scaler,
            ema_model=ema_model,
            cfg=cfg,
            device=device,
            stage_name=f"Epoch {global_epoch}/{cfg.total_epochs} [head-only]",
        )

        eval_model = ema_model.module
        style_val = evaluate_task(eval_model, loaders["style_val"], "style", style_eval_criterion, device, n_style, tta_flip=True)
        artist_val = evaluate_task(eval_model, loaders["artist_val"], "artist", artist_eval_criterion, device, n_artist, tta_flip=True)

        history.append({
            "epoch": global_epoch,
            "stage": "head-only",
            "lr": optimizer_s1.param_groups[0]["lr"],
            "train_loss": train_metrics["loss"],
            "train_artist_acc": train_metrics["artist_top1"],
            "train_artist_ce": train_metrics["artist_loss"],
            "train_style_distill": train_metrics["distill_loss"],
            "val_style_loss": style_val["loss"],
            "val_style_top1": style_val["top1"],
            "val_style_top5": style_val["top5"],
            "val_artist_loss": artist_val["loss"],
            "val_artist_top1": artist_val["top1"],
            "val_artist_top5": artist_val["top5"],
        })

        marker = ""
        if (artist_val["top1"] > best_artist_val) or (math.isclose(artist_val["top1"], best_artist_val) and style_val["top1"] > best_style_val):
            best_artist_val = artist_val["top1"]
            best_style_val = style_val["top1"]
            best_epoch = global_epoch
            torch.save({
                "model_state": eval_model.state_dict(),
                "config": asdict(cfg),
                "best_epoch": best_epoch,
                "best_val_artist_top1": best_artist_val,
                "best_val_style_top1": best_style_val,
                "n_style": n_style,
                "n_artist": n_artist,
                "model_name": cfg.model_name,
            }, output_ckpt_path)
            marker = " <- best"

        print(f"Epoch {global_epoch:3d} | artist_val_top1={artist_val['top1']:.4f} | style_val_top1={style_val['top1']:.4f}{marker}")

    # Stage 2
    if cfg.run_stage2_finetune and cfg.finetune_epochs > 0:
        print("\n" + "=" * 72)
        print("Stage 2 - Fine-tune + style distillation")
        print("=" * 72)
        unfreeze_last_style_blocks(model, cfg.unfreeze_last_blocks)
        optimizer_s2 = build_stage2_optimizer(model, cfg)
        total_steps_s2 = max(1, cfg.finetune_epochs * len(loaders["artist_train"]) // cfg.accumulate_steps)
        warmup_s2 = cfg.warmup_epochs * len(loaders["artist_train"]) // cfg.accumulate_steps
        scheduler_s2 = build_scheduler(optimizer_s2, warmup_steps=warmup_s2, total_steps=total_steps_s2)

        for _ in range(cfg.finetune_epochs):
            global_epoch += 1
            train_metrics = train_artist_epoch(
                model=model,
                teacher_model=teacher_model,
                loader=loaders["artist_train"],
                optimizer=optimizer_s2,
                scheduler=scheduler_s2,
                scaler=scaler,
                ema_model=ema_model,
                cfg=cfg,
                device=device,
                stage_name=f"Epoch {global_epoch}/{cfg.total_epochs} [fine-tune]",
            )

            eval_model = ema_model.module
            style_val = evaluate_task(eval_model, loaders["style_val"], "style", style_eval_criterion, device, n_style, tta_flip=True)
            artist_val = evaluate_task(eval_model, loaders["artist_val"], "artist", artist_eval_criterion, device, n_artist, tta_flip=True)

            history.append({
                "epoch": global_epoch,
                "stage": "fine-tune",
                "lr": optimizer_s2.param_groups[0]["lr"],
                "train_loss": train_metrics["loss"],
                "train_artist_acc": train_metrics["artist_top1"],
                "train_artist_ce": train_metrics["artist_loss"],
                "train_style_distill": train_metrics["distill_loss"],
                "val_style_loss": style_val["loss"],
                "val_style_top1": style_val["top1"],
                "val_style_top5": style_val["top5"],
                "val_artist_loss": artist_val["loss"],
                "val_artist_top1": artist_val["top1"],
                "val_artist_top5": artist_val["top5"],
            })

            marker = ""
            if (artist_val["top1"] > best_artist_val) or (math.isclose(artist_val["top1"], best_artist_val) and style_val["top1"] > best_style_val):
                best_artist_val = artist_val["top1"]
                best_style_val = style_val["top1"]
                best_epoch = global_epoch
                torch.save({
                    "model_state": eval_model.state_dict(),
                    "config": asdict(cfg),
                    "best_epoch": best_epoch,
                    "best_val_artist_top1": best_artist_val,
                    "best_val_style_top1": best_style_val,
                    "n_style": n_style,
                    "n_artist": n_artist,
                    "model_name": cfg.model_name,
                }, output_ckpt_path)
                marker = " <- best"

            print(f"Epoch {global_epoch:3d} | artist_val_top1={artist_val['top1']:.4f} | style_val_top1={style_val['top1']:.4f}{marker}")

    best_state = torch.load(output_ckpt_path, map_location=device)
    model.load_state_dict(best_state["model_state"])
    model = model.to(device)

    final_style_val = evaluate_task(model, loaders["style_val"], "style", style_eval_criterion, device, n_style, tta_flip=True, tta_scales=[0.85, 0.925, 1.075, 1.15])
    final_style_test = evaluate_task(model, loaders["style_test"], "style", style_eval_criterion, device, n_style, tta_flip=True, tta_scales=[0.85, 0.925, 1.075, 1.15])
    final_artist_val = evaluate_task(model, loaders["artist_val"], "artist", artist_eval_criterion, device, n_artist, tta_flip=True)
    final_artist_test = evaluate_task(model, loaders["artist_test"], "artist", artist_eval_criterion, device, n_artist, tta_flip=True)

    history_df = pd.DataFrame(history)
    history_path = project_root / cfg.history_csv
    history_path.parent.mkdir(parents=True, exist_ok=True)
    history_df.to_csv(history_path, index=False)

    summary_row = {
        "notebook": "wikiart_style_classification_test13_style_artist_warmstart.ipynb",
        "exists": True,
        "model_name": cfg.model_name,
        "val_loss": final_style_val["loss"],
        "val_top1": final_style_val["top1"],
        "val_top5": final_style_val["top5"],
        "test_loss": final_style_test["loss"],
        "test_top1": final_style_test["top1"],
        "test_top5": final_style_test["top5"],
        "best_val_top1": best_style_val,
        "best_epoch": float(best_epoch),
        "experiment": "test13",
        "saved_at_utc": datetime.datetime.utcnow().isoformat(),
        "val_style_acc": final_style_val["top1"],
        "val_artist_acc": final_artist_val["top1"],
        "val_emotion_acc": "",
        "test_style_acc": final_style_test["top1"],
        "test_artist_acc": final_artist_test["top1"],
        "test_emotion_acc": "",
        "best_combined_score": 0.5 * (final_style_val["top1"] + final_artist_val["top1"]),
    }

    append_summary_row(
        summary_path=project_root / cfg.summary_csv,
        previous_summary_path=project_root / cfg.previous_summary_csv,
        row=summary_row,
    )

    print("\n=== FINAL RESULTS (Test 13) ===")
    print(f"Best epoch: {best_epoch}")
    print(f"Style val/test top1: {final_style_val['top1']*100:.2f}% / {final_style_test['top1']*100:.2f}%")
    print(f"Artist val/test top1: {final_artist_val['top1']*100:.2f}% / {final_artist_test['top1']*100:.2f}%")
    print(f"Checkpoint: {output_ckpt_path}")
    print(f"History CSV: {history_path}")
    print(f"Summary CSV: {project_root / cfg.summary_csv}")


run_test13(cfg)

[style_train] Skipping 2 missing files
[artist_train] Skipping 2 missing files
Classes -> style: 27, artist: 23
Artist train samples: 13,344 (batches=1668)
Style val/test: 12,213 / 12,208
Artist val/test: 2,855 / 2,851
Loading warm-start: C:\Users\Thijs\Desktop\Ai Art Critic\models\wikiart_test12_vitlarge_448_best.pt
Warm-start loaded: missing=0 unexpected=0 coverage=100.00%

Stage 1 - Train artist head only


Epoch 1/30 [head-only]:   0%|          | 0/1668 [00:00<?, ?it/s]

Epoch   1 | artist_val_top1=0.0392 | style_val_top1=0.7831 <- best


Epoch 2/30 [head-only]:   0%|          | 0/1668 [00:00<?, ?it/s]

Epoch   2 | artist_val_top1=0.0571 | style_val_top1=0.7831 <- best


...

Epoch 29/30 [fine-tune]:   0%|          | 0/1668 [00:00<?, ?it/s]

Epoch  29 | artist_val_top1=0.8725 | style_val_top1=0.7843 <- best


Epoch 30/30 [fine-tune]:   0%|          | 0/1668 [00:00<?, ?it/s]

Epoch  30 | artist_val_top1=0.8767 | style_val_top1=0.7843 <- best

=== FINAL RESULTS (Test 13) ===
Best epoch: 30
Style val/test top1: 78.24% / 77.37%
Artist val/test top1: 87.67% / 87.97%
Checkpoint: C:\Users\Thijs\Desktop\Ai Art Critic\models\wikiart_test13_style_artist_warmstart_best.pt
History CSV: C:\Users\Thijs\Desktop\Ai Art Critic\models\results\wikiart_test13_style_artist_history.csv
Summary CSV: C:\Users\Thijs\Desktop\Ai Art Critic\models\results\wikiart_tests_1_to_13_summary.csv


C:\Users\Thijs\AppData\Local\Temp\ipykernel_27584\1884461770.py:213: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "saved_at_utc": datetime.datetime.utcnow().isoformat(),
